# RepairScopeX — E01 + E04 + E05 Consolidated Research Analysis
**CPU only · no new diffusion-model generations · reuse all existing saved evidence**

- **E01:** empirical evaluability taxonomy, category/stopping-condition cross-tabs and uncertainty.
- **E04:** paired 96→192-token reanalysis of the completed NB29G cases, without counting unfinished trials.
- **E05:** cohort feasibility and failure-yield sensitivity, including Bayesian uncertainty on expected discovery effort.

Includes submitted **P2 oracle audit** and **P3 label audit** summaries. They are *reported* not re-executed. All four input archives are embedded with SHA-256 provenance so Colab needs no uploads. Baseline records and changed-budget records remain separate.

Use **Runtime → Run all** on a CPU runtime. On completion, download `RSX_E01_E04_E05_RESULTS.zip`.

In [ ]:
import io, json, base64, zipfile, hashlib, csv, math, os, statistics
from pathlib import Path
from collections import Counter,defaultdict
import numpy as np
import pandas as pd
from scipy.stats import beta, chi2_contingency
from IPython.display import display
import matplotlib.pyplot as plt
OUTPUT=Path('/content/RSX_E01_E04_E05') if Path('/content').exists() else Path('/mnt/data/RSX_E01_E04_E05')
OUTPUT.mkdir(parents=True,exist_ok=True)
print('Output:',OUTPUT)

In [ ]:
SOURCE_HASHES={'NB29E': '792ae72c636bf50beff78187bdcc93312baa9996afea9d57423be0842893652a', 'NB29G': '4700c47569fc5d032beaa3f81af140b013e25db209d63b0898fdc7ab717f1f8e', 'P2': '8376df56b3d8f0889cd66ad6a8b803ed2c8ced5fbac44818e42480b795f01d6e', 'P3': 'b9d499a35d42322e36b5a83884f60599f8d44a1e9850b8e81d9d0efcdaaa9263'}
EMBEDDED_B64='''...'''

raw=base64.b64decode(EMBEDDED_B64)
with zipfile.ZipFile(io.BytesIO(raw)) as big:
    inputs={name.removesuffix('.zip'):big.read(name) for name in big.namelist()}
for name,blob in inputs.items():
    found=hashlib.sha256(blob).hexdigest()
    assert found==SOURCE_HASHES[name], f'{name} snapshot mismatch'
print('Verified sources:', {k:len(v) for k,v in inputs.items()})
def open_source(name):
    return zipfile.ZipFile(io.BytesIO(inputs[name]))
def json_zip(z,name):return json.loads(z.read(name))
def csv_zip(z,name):return pd.read_csv(io.BytesIO(z.read(name)))
with open_source('NB29E') as z:
    baseline=[json_zip(z,n) for n in z.namelist() if n.startswith('records/natural/') and n.endswith('.json')]
    bank=json_zip(z,'frozen_problem_bank.json')['problems']
    proto=json_zip(z,'protocol.json')
with open_source('NB29G') as z:
    g_records=[json_zip(z,n) for n in z.namelist() if n.startswith('records/') and n.endswith('.json')]
    g_plan=json_zip(z,'precommitted_plan.json')
with open_source('P2') as z:p2=json_zip(z,'summary.json');p2_manifest=json_zip(z,'stress_suite_manifest.json')
with open_source('P3') as z:p3=json_zip(z,'summary.json');p3_review=csv_zip(z,'prioritized_human_review.csv')
assert len({r['trial_id'] for r in baseline})==len(baseline)
assert len({r['trial_id'] for r in g_records})==len(g_records)
assert len(baseline)==272 and len(g_records)==32, 'Embedded snapshot counts differ from verified sources'
assert len(bank)==648
assert p2['independent_gold_checked']==648 and p3['recomputation_mismatches']==0
assert p3['source_n']==len(baseline)
BMAP={p['problem_id']:p for p in bank}
assert all(r['problem_id'] in BMAP for r in baseline+g_records)
print('Loaded',len(baseline),'original generations;',len(g_records),'length-192 generations;',len(bank),'frozen tasks')

## E01 — Evaluability taxonomy, uncertainty and stopping reasons
A definite semantic failure is not the same as an unanswered generation. Rates are descriptive, with Wilson intervals; exploratory tests should not be read as causal or confirmatory.

In [ ]:
def wilson(k,n,z=1.959963984540054):
    if n==0:return (float('nan'),float('nan'))
    p=k/n;d=1+z*z/n
    mid=(p+z*z/(2*n))/d
    radius=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return mid-radius,mid+radius

def outcome(r):
    x=r.get('evaluation',{})
    s=x.get('failure_type')
    if s in ('semantically_correct','definite_wrong','unanswered_or_ambiguous'):return s
    if x.get('eligible_semantic_failure'):return 'definite_wrong'
    if x.get('success'):return 'semantically_correct'
    return 'unanswered_or_ambiguous'
rows=[{'trial_id':r['trial_id'],'problem_id':r['problem_id'],'split':r['split'],'kind':r['kind'],
       'outcome':outcome(r),'stop_reason':r.get('stop_reason','missing'),
       'generated_tokens':r.get('generated_tokens',np.nan),
       'wall_seconds':r.get('wall_seconds',np.nan)} for r in baseline]
df=pd.DataFrame(rows)
assert dict(df.outcome.value_counts())=={'unanswered_or_ambiguous':142,'semantically_correct':122,'definite_wrong':8}
results=[]
for keys,group in [('ALL',df)]+[(f'{s}/{k}',g) for (s,k),g in df.groupby(['split','kind'])]:
    for category in ('semantically_correct','definite_wrong','unanswered_or_ambiguous'):
        k=int((group.outcome==category).sum());lo,hi=wilson(k,len(group))
        results.append({'cohort':keys,'outcome':category,'n':len(group),'count':k,'rate':k/len(group),'ci_low':lo,'ci_high':hi})
pd.DataFrame(results).to_csv(OUTPUT/'E01_wilson_rates.csv',index=False)
stop_table=pd.crosstab([df['kind'],df['stop_reason']],df['outcome'])
stop_table.to_csv(OUTPUT/'E01_stop_outcome_table.csv')
pd.crosstab(df['split'],[df['kind'],df['outcome']]).to_csv(OUTPUT/'E01_split_task_outcomes.csv')
df.to_csv(OUTPUT/'E01_original_record_inventory.csv',index=False)
print('E01 overall outcomes:',df.outcome.value_counts().to_dict())
display(pd.DataFrame(results).query("cohort=='ALL'"))
display(stop_table)
# Exploratory association only, expected counts may be small.
ct=pd.crosstab(df.stop_reason,df.outcome)
if ct.shape[0]>1 and ct.shape[1]>1:
    chi,p,_,expected=chi2_contingency(ct,correction=False)
    association={'chi_square':float(chi),'p_exploratory':float(p),'min_expected_cell':float(expected.min()),
                 'warning':'Observational association, not evidence that stopping condition caused the outcome.'}
else:association={'not_tested':'Insufficient stop/outcome diversity'}
(OUTPUT/'E01_association.json').write_text(json.dumps(association,indent=2))
ax=df.outcome.value_counts().reindex(['semantically_correct','definite_wrong','unanswered_or_ambiguous']).plot.bar(figsize=(8,4),rot=15)
ax.set_ylabel('Generation count');ax.set_title('Original V1 outcomes, n=272');plt.tight_layout()
plt.savefig(OUTPUT/'E01_outcome_distribution.png',dpi=150);plt.close()

## E04 — Paired length-budget reanalysis
Only cases with **both** a saved 96-token natural output and a completed 192-token retest enter the paired analysis. Missing planned cases remain missing. The length intervention is a changed decoding protocol, not a continuation of V1.

In [ ]:
by_id={r['problem_id']:r for r in baseline}
paired=[]; fresh=[]
for r in g_records:
    p=by_id.get(r['problem_id'])
    if r['group']=='paired_ambiguous':
        assert p is not None and outcome(p)=='unanswered_or_ambiguous', 'Pair mismatch'
        paired.append({'problem_id':r['problem_id'],'kind':r['kind'],'split':r['split'],
                       'old_outcome':outcome(p),'new_outcome':outcome(r),
                       'old_stop':p.get('stop_reason'),'new_stop':r.get('stop_reason'),
                       'old_tokens':p.get('generated_tokens'),'new_tokens':r.get('generated_tokens')})
    elif r['group']=='fresh_unused':fresh.append(r)
    else:raise ValueError('Unknown G group')
paired_df=pd.DataFrame(paired);paired_df.to_csv(OUTPUT/'E04_paired_transitions.csv',index=False)
trans=pd.crosstab(paired_df.old_outcome,paired_df.new_outcome)
trans.to_csv(OUTPUT/'E04_transition_matrix.csv')
assert len(paired)==24 and len(fresh)==8
recoveries=int((paired_df.new_outcome=='semantically_correct').sum())
lo,hi=wilson(recoveries,len(paired))
from scipy.stats import binomtest
# Two-directional evaluability transitions, if available, exact discordant sign test.
b=sum((x['old_outcome']=='unanswered_or_ambiguous' and x['new_outcome']!='unanswered_or_ambiguous') for x in paired)
c=sum((x['old_outcome']!='unanswered_or_ambiguous' and x['new_outcome']=='unanswered_or_ambiguous') for x in paired)
summary_e04={'total_plan':len(g_plan),'completed':len(g_records),'pending':len(g_plan)-len(g_records),
             'paired_complete':len(paired),'fresh_completed':len(fresh),
             'paired_correct_recovered':recoveries,'recover_fraction':recoveries/len(paired),
             'wilson_95':[lo,hi], 'discordant_b':b,'discordant_c':c,
             'exact_two_sided_p_descriptive':binomtest(min(b,c),b+c,p=.5).pvalue if b+c>0 else None,
             'warning':'Selected conditional sample; not population-wide evidence of length effectiveness.'}
(OUTPUT/'E04_summary.json').write_text(json.dumps(summary_e04,indent=2))
print('E04:',summary_e04)
display(trans)

## E05 — Feasibility and sensitivity
Each original V1 split/task stratum is assessed against its original precommitted quota. Bayesian Beta(1/2, 1/2) calculations describe uncertainty in the failure yield; they do **not** impute additional observed failures. Separate diagnostics mark deficits that exceed candidate capacity.

In [ ]:
BSEEN={r['problem_id'] for r in baseline}
original_quota={'calibration':4,'heldout':8}
rng=np.random.default_rng(2905)
feasible=[]
for (split,kind),g in df.groupby(['split','kind']):
    n=len(g);f=int((g.outcome=='definite_wrong').sum());target=original_quota[split];missing=max(0,target-f)
    remaining=sum(p['split']==split and p['kind']==kind and p['problem_id'] not in BSEEN for p in bank)
    lo,hi=wilson(f,n)
    draws=rng.beta(f+0.5,n-f+0.5,size=20000)
    projected=missing/draws if missing else np.zeros_like(draws)
    feasible.append({'split':split,'kind':kind,'attempts':n,'eligible':f,'yield':f/n,
       'yield_wilson_lo':lo,'yield_wilson_hi':hi,'required':target,'deficit':missing,
       'unused_original_bank_candidates':remaining,'bank_insufficient_even_if_all_fail':bool(missing>remaining),
       'projected_attempts_median':float(np.median(projected)),
       'projected_attempts_p90':float(np.quantile(projected,.9)),
       'predictive_new_failures_mean':float(np.mean(remaining*draws)),
       'note':'Exploratory posterior sensitivity; ignores selection effects, dependence and regime changes.'})
fdf=pd.DataFrame(feasible);fdf.to_csv(OUTPUT/'E05_feasibility_by_stratum.csv',index=False)
display(fdf[['split','kind','attempts','eligible','deficit','unused_original_bank_candidates','projected_attempts_median']])
# Sensitivity: expected successes under optimistic/pessimistic fixed rates.
sensitivity=[]
for _,row in fdf.iterrows():
    for p in (max(row['yield']/2,0.001),max(row['yield'],0.001),min(max(row['yield']*2,0.001),0.95)):
        sensitivity.append({'split':row['split'],'kind':row['kind'],'assumed_failure_rate':p,
                            'expected_new_attempts_to_fill_deficit':row['deficit']/p})
pd.DataFrame(sensitivity).to_csv(OUTPUT/'E05_rate_sensitivity.csv',index=False)

## Evidence reconciliation and exports
P2/P3 were already executed. This notebook verifies their signed input ZIPs and reports their findings, without treating synthetic mutations as observed model outcomes.

In [ ]:
assert p2['gold_mismatches']==0 and p2['independent_gold_checked']==648
assert p3['recomputation_mismatches']==0 and p3['source_n']==len(df)
assert p3['synthetic_mutations']==816
reconciliation={'E01':{'baseline_n':len(df),'observed_counts':df.outcome.value_counts().to_dict()},
                'E04':summary_e04,'E05':{'cohorts':len(fdf),'remaining_required':int(fdf.deficit.sum())},
                'P2':p2,'P3':p3,'input_sha256':SOURCE_HASHES,
                'interpretation':'No new model generations or repairs were performed. P2/P3 are prior validation results.'}
(OUTPUT/'SUMMARY.json').write_text(json.dumps(reconciliation,indent=2))
# P3 review queue remains an auditable data file; neither label nor outcome overwritten.
p3_review.to_csv(OUTPUT/'P3_existing_human_review_queue.csv',index=False)
with zipfile.ZipFile('/content/RSX_E01_E04_E05_RESULTS.zip' if Path('/content').exists() else '/mnt/data/RSX_E01_E04_E05_RESULTS.zip','w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUTPUT.iterdir()):
        if f.is_file():z.write(f,f.name)
print('Finished CPU analyses; saved:',list(sorted(x.name for x in OUTPUT.iterdir())))
print('RESULT ZIP ready:', '/content/RSX_E01_E04_E05_RESULTS.zip' if Path('/content').exists() else '/mnt/data/RSX_E01_E04_E05_RESULTS.zip')